In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.metrics import calinski_harabasz_score
import matplotlib.pyplot as plt
import ancestree

rules = {
    "raw data": [None],
    "scaling": ["raw data"],
    "embedding": ["scaling"],
    "clustering": ["embedding"]
}
triggers = ["scaling", "embedding", "clustering"]

store = ancestree.LineageStore("../assets/demo", rules=rules, gen_triggers=triggers)

/Users/js/Documents/coding_projects/ancestree/ancestree-venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
scalers = {"robust": RobustScaler(), "standard":StandardScaler()}
embedders = ["umap", "tsne", "pca"]
clusterers = ["kmeans", "hdbscan"]

with store.create_node("raw data") as root_node:
    iris = load_iris()
    df_raw = pd.DataFrame(iris.data, columns = iris.feature_names)
    df_raw.to_csv(root_node/"data.csv", index=False)

    # Scaling
    for s_name, s_obj in scalers.items():
        with store.create_node("scaling", parent=root_node) as s_node:
            scaled_data = s_obj.fit_transform(df_raw)
            pd.DataFrame(scaled_data).to_csv(s_node/"scaled.csv", index=False)

            # Dim reduction
            for e_type in embedders:
                with store.create_node("embedding", parent=s_node) as e_node:
                    if e_type == "pca":
                        embedder = PCA(n_components=2)
                    elif e_type == "tsne":
                        embedder = TSNE(n_components=2, perplexity=30)
                    elif e_type == "umap":
                        embedder = umap.UMAP(n_components=2)
                    
                    embedding = embedder.fit_transform(scaled_data)

                    np.save(e_node/"map.npy", embedding)
                    e_node.add_meta('method', e_type, 'text', 'Method')

                    # Clustering
                    for c_type in clusterers:
                        with store.create_node("clustering", parent=e_node) as c_node:
                            if c_type == "kmeans":
                                model = KMeans(n_clusters=3, n_init="auto")
                                labels = model.fit_predict(embedding)
                            elif c_type == "hdbscan":
                                model = HDBSCAN(min_cluster_size=5)
                                labels = model.fit_predict(embedding)
                            
                            score = calinski_harabasz_score(embedding, labels)
                            c_node.add_meta('CH_score', score, 'text', 'Metrics')

                            plt.figure(figsize=(6,4))
                            plt.scatter(embedding[:, 0], embedding[:,1], c=labels, cmap='turbo')
                            plt.savefig(c_node/"clustering_img.png")
                            plt.close()
                            c_node.add_meta('Final Results', str(c_node/"clustering_img.png"), 'image', 'Results')

/Users/js/Documents/coding_projects/ancestree/ancestree-venv/lib/python3.11/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/js/Documents/coding_projects/ancestree/ancestree-venv/lib/python3.11/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/js/Documents/coding_projects/ancestree/ancestree-venv/lib/python3.11/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default value of `copy` will change from False to True in 1.10. Explicitly set a value for `copy` to silence this warning.
  warn(
/Users/js/Documents/coding_projects/ancestree/ancestree-venv/lib/python3.11/site-packages/sklearn/cluster/_hdbscan/hdbscan.py:722: FutureWarning: The default

In [4]:
store.generate_web_graph()

Graph generated at ../assets/demo/interactive_pipeline.html


In [4]:
# Use lambda functions for complex querying

bad_nodes = store.find_node(CH_score = lambda x: x<1000)
bad_nodes

[Node = 0742ecbc, path = 0742ecbc, generation = 3,
 Node = 3e590d66, path = 3e590d66, generation = 3,
 Node = 7db27a26, path = 7db27a26, generation = 3,
 Node = 3dbb7f4b, path = 3dbb7f4b, generation = 3,
 Node = 2f20a805, path = 2f20a805, generation = 3]

In [6]:
# Chain search terms next to each other and supply using dictionary unpacking

search_vars = {'method':'pca', 'step_type':'embedding'}

nodes_of_interest = store.find_node(**search_vars)
nodes_of_interest

[Node = d1080161, path = d1080161, generation = 2,
 Node = 8ffa1019, path = 8ffa1019, generation = 2]

In [7]:
last_node = store.get_most_recent_node()

nodes_in_lineage= store.get_lineage(last_node)

for n in nodes_in_lineage:
    print(n.artifacts())

[PosixPath('3a53db0b/data.csv')]
[PosixPath('dc5fcc55/scaled.csv')]
[PosixPath('8ffa1019/map.npy')]
[PosixPath('2f20a805/clustering_img.png')]


In [8]:
# Full list of commands


# store = ancestree.LineageStore('ML_pipeline')

# store.find_node(step_type='embedding')

# store.get_node(node='0d33f882')

# store.get_lineage(node = '0d33f882')

# store.find_in_lineage(node='0d33f882', step_type='embedding')

# store.get_most_recent_node()

# store.get_child_nodes(node='0d33f882')

# store.from_parent(node = '0d33f882', filename='*.csv')

# store.rebuild_db_from_disk()